# Benign package metadata fetcher (npm + PyPI)
Builds benign rows with the **same columns/definitions** as `manifest_npm_malicious_combined.csv` and `manifest_pypi_malicious_combined.csv`,
sizes the benign set from the number of malicious rows, and writes benign + combined CSVs.

Run cells top to bottom. Fetching is checkpointed (`/content/benign_work`), so if Colab disconnects just re-run and it resumes.

In [ ]:
!pip -q install rapidfuzz tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.0 MB/s eta 0:00:00


In [ ]:
import os, re, io, json, time, random
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests, pandas as pd
from tqdm.auto import tqdm
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein

# ------------------------- CONFIG -------------------------
RATIO = {"npm": 3, "pypi": 3}   # benign packages per 1 malicious package (1 = balanced)
SEED = 42
TOP_N = 2000                     # size of the "top packages" list used for the typo_* features
TOP_FRAC_IN_BENIGN = 0.10        # share of benign drawn from the top-N popular packages (rest = random long tail)
MATCH_NPM_SCOPE_RATIO = True     # match the share of @scoped names in benign to the malicious npm set (~86%)
OVERSAMPLE = 1.4                 # extra candidates to cover 404s / skipped packages
WORKERS = 16
WORKDIR = "/content/benign_work"
os.makedirs(WORKDIR, exist_ok=True)
random.seed(SEED)
NOW = datetime.now(timezone.utc)

In [ ]:
# Upload both malicious CSVs (or place them in /content beforehand)
NPM_MAL, PYPI_MAL = "manifest_npm_malicious_combined.csv", "manifest_pypi_malicious_combined.csv"
if not (os.path.exists(NPM_MAL) and os.path.exists(PYPI_MAL)):
    from google.colab import files
    files.upload()

mal = {"npm": pd.read_csv(NPM_MAL), "pypi": pd.read_csv(PYPI_MAL)}
COLS = {k: list(v.columns) for k, v in mal.items()}
TARGET = {k: round(len(mal[k]) * RATIO[k]) for k in mal}

print(pd.DataFrame({"malicious": {k: len(v) for k, v in mal.items()},
                    "ratio (benign:mal)": RATIO,
                    "benign to fetch": TARGET}))

Saving manifest_npm_malicious_combined.csv to manifest_npm_malicious_combined.csv
Saving manifest_pypi_malicious_combined.csv to manifest_pypi_malicious_combined.csv
      malicious  ratio (benign:mal)  benign to fetch
npm        2833                   3             8499
pypi        188                   3              564


In [ ]:
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "malicious-pkg-dataset-research/1.0"
SESSION.mount("https://", requests.adapters.HTTPAdapter(pool_connections=WORKERS, pool_maxsize=WORKERS))

def get_json(url, headers=None, timeout=45, tries=4, max_bytes=40_000_000):
    for i in range(tries):
        try:
            r = SESSION.get(url, headers=headers, timeout=timeout, stream=True)
            if r.status_code == 404:
                return None
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i + random.random()); continue
            r.raise_for_status()
            if int(r.headers.get("content-length") or 0) > max_bytes:
                return None
            buf, size = io.BytesIO(), 0
            for chunk in r.iter_content(1 << 16):
                size += len(chunk)
                if size > max_bytes:
                    return None
                buf.write(chunk)
            return json.loads(buf.getvalue())
        except (requests.RequestException, ValueError):
            time.sleep(2 ** i)
    return None

def parse_ts(s):
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(timezone.utc)
    except Exception:
        return None

def norm_py(n):  # PEP 503
    return re.sub(r"[-_.]+", "-", n).lower()

## 1. Reference lists: top-N popular packages (for the `typo_*` features) and candidate pools

In [ ]:
def top_pypi(n):
    j = get_json("https://hugovk.github.io/top-pypi-packages/top-pypi-packages.min.json")
    rows = sorted(j["rows"], key=lambda r: -r["download_count"])[:n]
    return [norm_py(r["project"]) for r in rows]

def top_npm(n):
    names = []
    for frm in range(0, n + 250, 250):
        j = get_json("https://registry.npmjs.org/-/v1/search?text=not:deprecated"
                     f"&popularity=1.0&quality=0.0&maintenance=0.0&size=250&from={frm}")
        if not j or not j.get("objects"):
            break
        names += [o["package"]["name"] for o in j["objects"]]
        if len(names) >= n:
            break
    names = list(dict.fromkeys(names))
    if len(names) < 0.9 * n:   # fallback: wooorm/npm-high-impact
        try:
            txt = SESSION.get("https://raw.githubusercontent.com/wooorm/npm-high-impact/main/lib/top.js", timeout=60).text
            alt = re.findall(r"'([^']+)'|\"([^\"]+)\"", txt)
            alt = [a or b for a, b in alt if (a or b) and " " not in (a or b)]
            names = list(dict.fromkeys(names + alt))
        except requests.RequestException:
            pass
    return names[:n]

TOP = {"pypi": top_pypi(TOP_N), "npm": top_npm(TOP_N)}
print({k: len(v) for k, v in TOP.items()}, TOP["npm"][:5], TOP["pypi"][:5])

{'pypi': 2000, 'npm': 2000} ['gitbook-plugin-npmsearchlist', 'semver', 'minimatch', 'debug', 'ansi-styles'] ['boto3', 'packaging', 'typing-extensions', 'certifi', 'idna']


In [ ]:
def pool_pypi():
    j = get_json("https://pypi.org/simple/", headers={"Accept": "application/vnd.pypi.simple.v1+json"}, max_bytes=400_000_000)
    return [p["name"] for p in j["projects"]]

def pool_npm():
    # Preferred: full name list. Fallback: sample the search API with random tokens.
    try:
        j = get_json("https://raw.githubusercontent.com/nice-registry/all-the-package-names/master/names.json",
                     timeout=300, max_bytes=800_000_000)
        if j:
            return j
    except Exception:
        pass
    print("names.json unavailable -> falling back to search-API sampling")
    rng, names = random.Random(SEED), set()
    alphabet = "abcdefghijklmnopqrstuvwxyz0123456789"
    for _ in tqdm(range(400)):
        tok = "".join(rng.choice(alphabet) for _ in range(3))
        j = get_json(f"https://registry.npmjs.org/-/v1/search?text={tok}&size=250")
        if j:
            names.update(o["package"]["name"] for o in j.get("objects", []))
    return sorted(names)

POOL = {"pypi": pool_pypi(), "npm": pool_npm()}
print({k: len(v) for k, v in POOL.items()})

names.json unavailable -> falling back to search-API sampling


  0%|          | 0/400 [00:00<?, ?it/s]

{'pypi': 902446, 'npm': 9745}


In [ ]:
def pick_candidates(reg):
    rng = random.Random(SEED)
    n_cand = int(TARGET[reg] * OVERSAMPLE)
    excl = set(mal[reg].package_name) | ({norm_py(x) for x in mal[reg].package_name} if reg == "pypi" else set())
    top = TOP[reg]; top_set = set(top)
    top_pick = rng.sample([t for t in top if t not in excl], min(int(n_cand * TOP_FRAC_IN_BENIGN), len(top)))
    n_tail = n_cand - len(top_pick)
    tail = [x for x in POOL[reg] if x not in top_set and x not in excl and (norm_py(x) not in top_set if reg == "pypi" else True)]
    if reg == "npm" and MATCH_NPM_SCOPE_RATIO:
        sr = mal["npm"].package_name.str.startswith("@").mean()
        sc = [x for x in tail if x.startswith("@")]; un = [x for x in tail if not x.startswith("@")]
        n_sc = round(n_tail * sr)
        tail_pick = rng.sample(sc, min(n_sc, len(sc))) + rng.sample(un, min(n_tail - n_sc, len(un)))
    else:
        tail_pick = rng.sample(tail, min(n_tail, len(tail)))
    out = [(n, "registry_top2000") for n in top_pick] + [(n, "registry_random") for n in tail_pick]
    rng.shuffle(out)
    return out

CAND = {r: pick_candidates(r) for r in ("npm", "pypi")}
print({k: len(v) for k, v in CAND.items()})

{'npm': 6669, 'pypi': 789}


## 2. Feature extraction (mirrors the malicious CSV columns)

In [ ]:
# ---------------- npm ----------------
INSTALL_HOOKS = ("preinstall", "install", "postinstall")
# NOTE: pattern set is my reconstruction of `dtm_suspicious_hook_api_hits` (malicious values range 0-3).
# If you have your original feature script, paste its patterns here so benign/malicious are computed identically.
SUSPICIOUS = [re.compile(p, re.I) for p in [
    r"\bcurl\b|\bwget\b",
    r"\beval\b|new Function",
    r"child_process|\bexec(Sync)?\b|\bspawn\b",
    r"base64|Buffer\.from",
    r"https?://|\bnc\b|netcat|/dev/tcp",
    r"powershell|cmd(\.exe)?\b|\b(ba)?sh -c",
    r"process\.env|\.ssh|\.npmrc|/etc/passwd",
]]

def npm_features(name, doc):
    versions = doc.get("versions") or {}
    if not versions:
        return None
    tags = doc.get("dist-tags") or {}
    latest = versions.get(tags.get("latest")) or versions[list(versions)[-1]]
    if any(v.endswith("-security") for v in versions) or "security holding package" in str(doc.get("description", "")).lower():
        return None                                  # npm's takedown placeholder -> not a real benign package
    desc = doc.get("description") or latest.get("description") or ""
    desc = desc if isinstance(desc, str) else ""
    readme = doc.get("readme") or ""
    readme = readme if isinstance(readme, str) else ""
    kw = latest.get("keywords") or doc.get("keywords") or []
    kw_n = len([k for k in re.split(r"[,\s]+", kw) if k]) if isinstance(kw, str) else len(kw)
    n_hooks, hits = 0, set()
    max_deps = 0
    for v in versions.values():
        scripts = v.get("scripts") or {}
        if isinstance(scripts, dict):
            for h in INSTALL_HOOKS:
                if scripts.get(h):
                    n_hooks += 1
                    for i, pat in enumerate(SUSPICIOUS):
                        if pat.search(str(scripts[h])):
                            hits.add(i)
        deps = v.get("dependencies") or {}
        max_deps = max(max_deps, len(deps) if isinstance(deps, (dict, list)) else 0)
    t = doc.get("time") or {}
    created = parse_ts(t.get("created", ""))
    vt = [parse_ts(t[v]) for v in versions if v in t]
    vt = [x for x in vt if x]
    return {
        "package_name": name, "registry": "npm", "label": "benign",
        "etm_description_length": min(len(desc), 255),          # malicious set is capped at 255
        "etm_readme_length": len(readme),
        "etm_has_readme": int(len(readme) > 0),
        "etm_has_homepage": int(bool(latest.get("homepage") or doc.get("homepage"))),
        "etm_keyword_count": kw_n,
        "etm_license_declared": int(bool(latest.get("license") or doc.get("license"))),
        "etm_maintainer_count": len(doc.get("maintainers") or []),
        "dtm_num_versions": len(versions),
        "dtm_age_days_since_created": (NOW - created).days if created else -1,
        "dtm_lifespan_days": (max(vt) - min(vt)).days if vt else -1,
        "dtm_max_dependency_count": max_deps,
        "dtm_has_lifecycle_hook": int(n_hooks > 0),
        "dtm_num_lifecycle_hooks": n_hooks,
        "dtm_suspicious_hook_api_hits": len(hits),
    }

def fetch_npm(item):
    name, src = item
    doc = get_json("https://registry.npmjs.org/" + name.replace("/", "%2F"))
    row = npm_features(name, doc) if doc else None
    if row: row["source"] = src
    return name, row

# ---------------- PyPI ----------------
def pypi_features(name, doc):
    info, rel = doc.get("info") or {}, doc.get("releases") or {}
    rel = {v: f for v, f in rel.items() if f}                   # releases that actually have files
    if not rel:
        return None
    desc, summ = info.get("description") or "", info.get("summary") or ""
    urls = info.get("project_urls") or {}
    cls = info.get("classifiers") or []
    kw = info.get("keywords") or ""
    kw_n = len([k for k in re.split(r"[,\s]+", kw) if k]) if isinstance(kw, str) else len(kw)
    files = {f["filename"] for fl in rel.values() for f in fl}
    ups = [parse_ts(f.get("upload_time_iso_8601") or "") for fl in rel.values() for f in fl]
    ups = [u for u in ups if u]
    has_home = bool(info.get("home_page")) or any(k.lower() in ("homepage", "home") for k in urls)
    return {
        "package_name": name, "registry": "pypi", "label": "benign",
        "etm_description_length": len(desc),
        "etm_summary_length": len(summ),
        "etm_has_readme": int(len(desc) > 20),                  # threshold inferred from the malicious CSV
        "etm_has_homepage": int(has_home),
        "etm_num_project_urls": len(urls),
        "etm_num_classifiers": len(cls),
        "etm_keyword_count": kw_n,
        "etm_license_declared": int(bool(info.get("license")) or any(c.startswith("License ::") for c in cls)),
        "etm_author_count": int(bool(info.get("author") or info.get("author_email"))) + int(bool(info.get("maintainer") or info.get("maintainer_email"))),
        "dtm_num_versions": len(rel),
        "dtm_age_days_since_first_upload": (NOW - min(ups)).days if ups else -1,
        "dtm_lifespan_days": (max(ups) - min(ups)).days if ups else -1,
        "dtm_num_distinct_files": len(files),
        "dtm_requires_dist_count": len(info.get("requires_dist") or []),
        "dtm_has_vulnerabilities_on_record": int(bool(doc.get("vulnerabilities"))),
        "dtm_yanked": int(bool(info.get("yanked"))),
    }

def fetch_pypi(item):
    name, src = item
    doc = get_json(f"https://pypi.org/pypi/{name}/json")
    row = pypi_features(name, doc) if doc else None
    if row: row["source"] = src
    return name, row

In [ ]:
def run_fetch(reg, fetch_fn, batch=WORKERS * 25):
    ckpt = f"{WORKDIR}/{reg}_benign.jsonl"
    done = {}
    if os.path.exists(ckpt):
        for line in open(ckpt):
            d = json.loads(line); done[d["name"]] = d["row"]
    todo = [c for c in CAND[reg] if c[0] not in done]
    n_ok = lambda: sum(1 for r in done.values() if r)
    pbar = tqdm(total=TARGET[reg], initial=min(n_ok(), TARGET[reg]), desc=f"{reg} benign")
    with open(ckpt, "a") as f, ThreadPoolExecutor(WORKERS) as ex:
        for i in range(0, len(todo), batch):
            if n_ok() >= TARGET[reg]:
                break
            futs = [ex.submit(fetch_fn, it) for it in todo[i:i + batch]]
            for fu in as_completed(futs):
                name, row = fu.result()
                done[name] = row
                f.write(json.dumps({"name": name, "row": row}) + "\n"); f.flush()
                if row: pbar.update(1)
    pbar.close()
    rows = [r for r in done.values() if r]
    # keep top-N share close to TOP_FRAC_IN_BENIGN, then trim to target
    random.Random(SEED).shuffle(rows)
    if len(rows) < TARGET[reg]:
        print(f"WARNING {reg}: only {len(rows)}/{TARGET[reg]} fetched - raise OVERSAMPLE and re-run this cell.")
    return rows[:TARGET[reg]]

benign_rows = {}
benign_rows["pypi"] = run_fetch("pypi", fetch_pypi)
benign_rows["npm"] = run_fetch("npm", fetch_npm)      # slower: npm packuments are large

pypi benign:   0%|          | 0/564 [00:00<?, ?it/s]

npm benign:   0%|          | 0/8499 [00:00<?, ?it/s]

WARNING npm: only 6666/8499 fetched - raise OVERSAMPLE and re-run this cell.


## 3. Typo features (same definition as the malicious CSVs)
`typo_min_distance_to_top2000` = min Levenshtein distance to the top-2000 list, `is_exact_name_collision` = distance 0, `is_likely_typosquat` = distance 1 or 2 (verified against your files).

In [ ]:
def add_typo(df, reg):
    tops = TOP[reg]
    tn = [norm_py(t) for t in tops] if reg == "pypi" else tops
    dists, closest = [], []
    for nm in tqdm(df.package_name, desc=f"typo {reg}"):
        q = norm_py(nm) if reg == "pypi" else nm
        _, d, idx = process.extractOne(q, tn, scorer=Levenshtein.distance)
        dists.append(int(d)); closest.append(tops[idx])
    df["typo_min_distance_to_top2000"] = dists
    df["typo_closest_package"] = closest
    df["is_exact_name_collision"] = (df.typo_min_distance_to_top2000 == 0).astype(int)
    df["is_likely_typosquat"] = df.typo_min_distance_to_top2000.isin([1, 2]).astype(int)
    return df

final = {}
for reg in ("npm", "pypi"):
    ben = add_typo(pd.DataFrame(benign_rows[reg]), reg)[COLS[reg]]     # identical column order as malicious CSV
    ben.to_csv(f"manifest_{reg}_benign.csv", index=False)
    comb = pd.concat([mal[reg], ben], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    comb.to_csv(f"dataset_{reg}_combined.csv", index=False)
    final[reg] = (ben, comb)
    print(reg, "->", comb.label.value_counts().to_dict())

typo npm:   0%|          | 0/6666 [00:00<?, ?it/s]

npm -> {'benign': 6666, 'malicious': 2833}


typo pypi:   0%|          | 0/564 [00:00<?, ?it/s]

pypi -> {'benign': 564, 'malicious': 188}


## 4. Leakage audit — read this before training
Malicious packages were mostly collected after takedown, so some fields are placeholders (`-1`, or always 0). Real benign values in those columns
would let a model separate classes trivially without learning anything about maliciousness.

In [ ]:
for reg in ("npm", "pypi"):
    ben, comb = final[reg]
    m = mal[reg]
    num = [c for c in COLS[reg] if c.startswith(("etm_", "dtm_"))]
    const = [c for c in num if m[c].nunique() == 1 and ben[c].nunique() > 1]
    minus1 = [c for c in num if (m[c] == -1).mean() > 0.3 and (ben[c] == -1).mean() < 0.05]
    print(f"[{reg}] constant in malicious but varying in benign -> drop or ignore:", const)
    print(f"[{reg}] mostly -1 (missing) in malicious, real values in benign  -> drop or treat as missing:", minus1)
    if reg == "pypi":
        print("   (PyPI: this affects the 'maloss' rows; 'backstabber' rows do have real ages)")
    print("   scoped-name share  malicious=%.2f benign=%.2f" % (
        m.package_name.str.startswith("@").mean(), ben.package_name.str.startswith("@").mean()) if reg == "npm" else "")

[npm] constant in malicious but varying in benign -> drop or ignore: ['etm_readme_length', 'etm_has_readme', 'dtm_age_days_since_created', 'dtm_lifespan_days']
[npm] mostly -1 (missing) in malicious, real values in benign  -> drop or treat as missing: ['dtm_age_days_since_created', 'dtm_lifespan_days']
   scoped-name share  malicious=0.86 benign=0.65
[pypi] constant in malicious but varying in benign -> drop or ignore: ['dtm_has_vulnerabilities_on_record', 'dtm_yanked']
[pypi] mostly -1 (missing) in malicious, real values in benign  -> drop or treat as missing: ['dtm_age_days_since_first_upload', 'dtm_lifespan_days']
   (PyPI: this affects the 'maloss' rows; 'backstabber' rows do have real ages)



In [ ]:
# Optional: download outputs
from google.colab import files
for f in ["manifest_npm_benign.csv", "manifest_pypi_benign.csv", "dataset_npm_combined.csv", "dataset_pypi_combined.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>